In [1]:
# Complete Spark Session Reset and Cleanup
print("🧹 Complete Spark Session Reset and Cleanup...")

import os
import sys
import time
import gc
import subprocess

def force_cleanup_spark():
    """Force cleanup of all Spark processes and connections"""
    try:
        # Clear Python-level Spark references
        if 'pyspark' in sys.modules:
            print("   🔄 Clearing PySpark modules...")
            
            # Try to stop any active sessions
            try:
                from pyspark.sql import SparkSession
                from pyspark import SparkContext
                
                # Get and stop active session
                active_session = SparkSession.getActiveSession()
                if active_session:
                    active_session.stop()
                    print("   ✓ Stopped active SparkSession")
                
                # Get and stop active context
                active_context = SparkContext._active_spark_context
                if active_context:
                    active_context.stop()
                    print("   ✓ Stopped active SparkContext")
                    
            except Exception as e:
                print(f"   ⚠️ Session cleanup: {str(e)}")
            
            # Clear class-level references
            try:
                SparkSession._instantiatedSession = None
                SparkContext._active_spark_context = None
                SparkContext._gateway = None
                SparkContext._jvm = None
                print("   ✓ Cleared Spark class references")
            except:
                pass
        
        # Force garbage collection
        gc.collect()
        time.sleep(2)
        
        # Kill any leftover Java processes (optional, uncomment if needed)
        # try:
        #     subprocess.run("taskkill /F /IM java.exe", shell=True, capture_output=True)
        #     print("   ✓ Killed Java processes")
        # except:
        #     pass
        
        print("   ✅ Spark cleanup completed")
        
    except Exception as e:
        print(f"   ⚠️ Cleanup error (non-critical): {str(e)}")

# Perform cleanup
force_cleanup_spark()

print("=" * 60)

🧹 Complete Spark Session Reset and Cleanup...
   ✅ Spark cleanup completed
   ✅ Spark cleanup completed


# PySpark MLlib Multi-Model Drug Interaction Prediction

This notebook implements and compares three different machine learning models using PySpark MLlib for drug interaction safety prediction:

1. **Logistic Regression** - Binary classification with regularization
2. **Random Forest Classifier** - Ensemble method for robust predictions
3. **Gradient Boosted Trees (GBT)** - Advanced boosting algorithm

## Key Features:
- **PySpark MLlib**: Distributed machine learning at scale
- **HDFS Integration**: Direct data loading from HDFS
- **Comprehensive Evaluation**: Multiple metrics and visualization
- **Model Comparison**: Side-by-side performance analysis

## Dataset:
- Source: HDFS path `hdfs://localhost:9000/output/combined_dataset_complete.csv`
- Features: Drug combinations, dosage information, safety labels
- Complete dataset processing with PySpark

In [15]:
# Section 1: Environment Setup and Imports
import warnings
warnings.filterwarnings('ignore')

# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit, concat_ws, isnan, count, mean, stddev
from pyspark.sql.types import DoubleType

# PySpark ML imports
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

# Visualization and metrics
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from datetime import datetime

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✓ All libraries imported successfully!")
print(f"⏰ Notebook started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ All libraries imported successfully!
⏰ Notebook started at: 2025-10-08 20:38:51


In [5]:
# Section 2: Initialize Spark Session (Optimized for 16GB RAM - Conservative)
print("🚀 Initializing Spark Session (Optimized for 16GB System)...")

import os
import sys
from pathlib import Path
import time
import gc

# Advanced Java configuration for conda environments
def setup_java_environment():
    """Setup Java environment for PySpark in conda environment"""
    print("   🔧 Configuring Java environment...")
    
    # Get conda environment prefix
    conda_prefix = os.environ.get('CONDA_PREFIX', '')
    print(f"   CONDA_PREFIX: {conda_prefix}")
    
    # Try multiple Java home locations
    java_home_candidates = []
    
    if conda_prefix:
        # Primary location: conda env's JVM
        java_home_candidates.extend([
            os.path.join(conda_prefix, 'Library', 'lib', 'jvm'),
            os.path.join(conda_prefix, 'Library'),
            os.path.join(conda_prefix, 'jre'),
            os.path.join(conda_prefix, 'lib', 'jvm')
        ])
    
    # System Java locations
    java_home_candidates.extend([
        os.environ.get('JAVA_HOME', ''),
        'C:\\Program Files\\Java\\jdk-11',
        'C:\\Program Files\\Java\\jre1.8.0_*',
        'C:\\Program Files (x86)\\Java\\jdk-11',
    ])
    
    # Find working Java home
    for java_home in java_home_candidates:
        if not java_home or not os.path.exists(java_home):
            continue
            
        # Check for java executable
        java_exe_paths = [
            os.path.join(java_home, 'bin', 'java.exe'),
            os.path.join(java_home, 'bin', 'java'),
        ]
        
        for java_exe in java_exe_paths:
            if os.path.exists(java_exe):
                os.environ['JAVA_HOME'] = java_home
                print(f"   ✓ JAVA_HOME set to: {java_home}")
                print(f"   ✓ Java executable: {java_exe}")
                return java_home
    
    print("   ⚠️ Could not find Java installation")
    return None

# Setup Java environment
java_home = setup_java_environment()

# Configure PySpark environment variables
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Clean environment variables
env_vars_to_remove = ['PYSPARK_SUBMIT_ARGS', 'SPARK_HOME']
for var in env_vars_to_remove:
    if var in os.environ:
        del os.environ[var]
        print(f"   🧹 Removed {var} from environment")

# Conservative memory configuration for 16GB system (using 12GB for Spark to be safe)
print("   💾 Configuring conservative memory for 16GB system (12GB for Spark, 4GB for OS)...")
TOTAL_SPARK_MEMORY = 12  # GB - Conservative allocation
DRIVER_MEMORY = 6        # GB - Reduced for stability
EXECUTOR_MEMORY = 4      # GB - Reduced for stability  
NUM_CORES = 2           # Conservative core count

try:
    print(f"   🔄 Creating SparkSession with conservative configuration...")
    print(f"      Driver Memory: {DRIVER_MEMORY}GB")
    print(f"      Executor Memory: {EXECUTOR_MEMORY}GB") 
    print(f"      Total Spark Memory: ~{DRIVER_MEMORY + EXECUTOR_MEMORY}GB")
    print(f"      CPU Cores: {NUM_CORES}")
    
    # Import Spark modules fresh
    from pyspark.sql import SparkSession
    from pyspark import SparkConf
    
    # Create SparkSession with conservative settings
    spark = SparkSession.builder \
        .appName("DrugInteractionMLlib") \
        .master(f"local[{NUM_CORES}]") \
        .config("spark.driver.memory", f"{DRIVER_MEMORY}g") \
        .config("spark.executor.memory", f"{EXECUTOR_MEMORY}g") \
        .config("spark.driver.maxResultSize", "2g") \
        .config("spark.sql.shuffle.partitions", str(NUM_CORES * 2)) \
        .config("spark.driver.host", "localhost") \
        .config("spark.driver.bindAddress", "127.0.0.1") \
        .config("spark.network.timeout", "300s") \
        .config("spark.executor.heartbeatInterval", "30s") \
        .config("spark.rpc.askTimeout", "300s") \
        .config("spark.serializer", "org.apache.spark.serializer.JavaSerializer") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
        .config("spark.storage.memoryFraction", "0.6") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .getOrCreate()

    # Set log level to reduce verbosity
    spark.sparkContext.setLogLevel("WARN")
    
    # Test the connection with a simple operation
    print("   🧪 Testing Spark connectivity...")
    test_df = spark.range(10000).repartition(NUM_CORES)
    count = test_df.count()
    test_df.unpersist()  # Clean up immediately
    print(f"   ✓ Connectivity test successful (processed {count:,} records)")

    print("✅ Spark Session initialized successfully!")
    print(f"   Spark Version: {spark.version}")
    print(f"   Master: {spark.sparkContext.master}")
    print(f"   App Name: {spark.sparkContext.appName}")
    
    try:
        java_version = spark.sparkContext._gateway.jvm.System.getProperty('java.version')
        print(f"   Java Version: {java_version}")
    except Exception:
        print("   Java Version: Unable to retrieve")
    
    print(f"   Java Home: {os.environ.get('JAVA_HOME', 'Not Set')}")
    print(f"   Python Version: {sys.version.split()[0]}")
    print(f"   Memory Configuration:")
    print(f"      ├─ Driver Memory: {DRIVER_MEMORY}GB")
    print(f"      ├─ Executor Memory: {EXECUTOR_MEMORY}GB")
    print(f"      ├─ Total Allocated: {DRIVER_MEMORY + EXECUTOR_MEMORY}GB")
    print(f"      └─ System Available: 6GB+ remaining")
    
except Exception as e:
    print(f"❌ Error initializing Spark: {str(e)}")
    print(f"   Error type: {type(e).__name__}")
    
    print("\n🔧 Troubleshooting Info:")
    print(f"   Python executable: {sys.executable}")
    print(f"   CONDA_PREFIX: {os.environ.get('CONDA_PREFIX', 'Not Set')}")
    print(f"   JAVA_HOME: {os.environ.get('JAVA_HOME', 'Not Set')}")
    print(f"   PYSPARK_PYTHON: {os.environ.get('PYSPARK_PYTHON', 'Not Set')}")
    
    print("\n🔄 Trying ultra-minimal configuration...")
    try:
        # Wait and clear
        time.sleep(3)
        gc.collect()
        
        # Ultra minimal configuration
        spark = SparkSession.builder \
            .appName("DrugInteractionMLlib") \
            .master("local[1]") \
            .config("spark.driver.memory", "4g") \
            .config("spark.executor.memory", "2g") \
            .config("spark.driver.host", "localhost") \
            .config("spark.driver.bindAddress", "127.0.0.1") \
            .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
            .config("spark.sql.shuffle.partitions", "2") \
            .config("spark.network.timeout", "120s") \
            .config("spark.serializer", "org.apache.spark.serializer.JavaSerializer") \
            .getOrCreate()
        
        spark.sparkContext.setLogLevel("ERROR")
        
        # Minimal test
        test_count = spark.range(100).count()
        print(f"✅ Spark initialized with ultra-minimal config! (test: {test_count})")
        print(f"   Memory: 4GB Driver + 2GB Executor = 6GB total")
        
    except Exception as e2:
        print(f"❌ Ultra-minimal configuration failed: {str(e2)}")
        print("\n💡 Solutions:")
        print("   1. Restart the Jupyter kernel: Kernel → Restart")
        print("   2. Close memory-intensive applications")
        print("   3. Restart VS Code completely")
        print("   4. Check Task Manager for Java processes and end them")
        print("   5. Try restarting your computer if the issue persists")
        raise

print("\n" + "="*60)

🚀 Initializing Spark Session (Optimized for 16GB System)...
   🔧 Configuring Java environment...
   CONDA_PREFIX: C:\Users\tendo\miniconda3\envs\torch_new
   ✓ JAVA_HOME set to: C:\Users\tendo\miniconda3\envs\torch_new\Library\lib\jvm
   ✓ Java executable: C:\Users\tendo\miniconda3\envs\torch_new\Library\lib\jvm\bin\java.exe
   🧹 Removed PYSPARK_SUBMIT_ARGS from environment
   🧹 Removed SPARK_HOME from environment
   💾 Configuring conservative memory for 16GB system (12GB for Spark, 4GB for OS)...
   🔄 Creating SparkSession with conservative configuration...
      Driver Memory: 6GB
      Executor Memory: 4GB
      Total Spark Memory: ~10GB
      CPU Cores: 2
   🧪 Testing Spark connectivity...
   🧪 Testing Spark connectivity...
   ✓ Connectivity test successful (processed 10,000 records)
✅ Spark Session initialized successfully!
   Spark Version: 3.5.3
   Master: local[2]
   App Name: DrugInteractionMLlib
   Java Version: 11.0.27
   Java Home: C:\Users\tendo\miniconda3\envs\torch_new\L

In [6]:
# Section 3: Load Data from HDFS
print("📊 Loading Drug Interaction Dataset from HDFS...")
print("   Source: hdfs://localhost:9000/output/combined_dataset_complete.csv")

hdfs_path = "hdfs://localhost:9000/output/combined_dataset_complete.csv"

try:
    # Load data from HDFS
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(hdfs_path)
    
    print(f"   ✓ Dataset loaded successfully!")
    print(f"   Total records: {df.count():,}")
    print(f"   Total columns: {len(df.columns)}")
    
    # Display schema
    print("\n📋 Dataset Schema:")
    df.printSchema()
    
    # Display sample data
    print("\n📝 Sample Records:")
    df.show(5, truncate=False)
    
    # Show statistics
    print("\n📈 Dataset Statistics:")
    df.groupBy("safety_label").count().show()
    
except Exception as e:
    print(f"   ❌ Error loading dataset: {str(e)}")
    print("   Please ensure HDFS is running and the dataset exists at the specified path")
    raise

print("\n" + "="*60)

📊 Loading Drug Interaction Dataset from HDFS...
   Source: hdfs://localhost:9000/output/combined_dataset_complete.csv
   ✓ Dataset loaded successfully!
   ✓ Dataset loaded successfully!
   Total records: 20,482,172
   Total columns: 16

📋 Dataset Schema:
root
 |-- subject_id: integer (nullable = true)
 |-- doses_per_24_hrs: string (nullable = true)
 |-- drug1: string (nullable = true)
 |-- drug2: string (nullable = true)
 |-- drug3: string (nullable = true)
 |-- drug4: string (nullable = true)
 |-- drug5: string (nullable = true)
 |-- drug6: string (nullable = true)
 |-- drug7: string (nullable = true)
 |-- drug8: string (nullable = true)
 |-- drug9: string (nullable = true)
 |-- drug10: string (nullable = true)
 |-- safety_label: string (nullable = true)
 |-- total_drugs: integer (nullable = true)
 |-- has_dosage_info: integer (nullable = true)
 |-- drug_combination_id: string (nullable = true)


📝 Sample Records:
+----------+----------------+----------------------------+-------------

In [16]:
# Section 4: Data Preprocessing
print("🔄 Preprocessing data for MLlib models...")

# Import missing functions  
from pyspark.sql.functions import when, sum, coalesce
from pyspark.sql.functions import col  # Re-import col to avoid naming conflicts
from pyspark.sql.types import DoubleType  # Import DoubleType

# Identify drug columns (drug1 through drug10)
drug_columns = [col_name for col_name in df.columns if col_name.startswith('drug') and col_name[4:].isdigit()]
print(f"   Found {len(drug_columns)} drug columns: {drug_columns}")

# Create a clean dataset with non-null values
df_clean = df.na.drop(subset=['safety_label'])

# Convert safety_label to numeric (0 for safe, 1 for unsafe)
df_clean = df_clean.withColumn(
    "label",
    when(col("safety_label") == "unsafe", 1.0).otherwise(0.0)
)

print(f"   ✓ Label encoding complete (safe=0, unsafe=1)")

# Create numerical features
# Handle total_drugs if it exists
if 'total_drugs' in df_clean.columns:
    df_clean = df_clean.withColumn('total_drugs', col('total_drugs').cast(DoubleType()))
else:
    # Count non-null drugs
    from pyspark.sql.functions import coalesce
    df_clean = df_clean.withColumn(
        'total_drugs',
        sum([when(col(drug_col).isNotNull(), 1).otherwise(0) for drug_col in drug_columns])
    )

# Handle doses_per_24_hrs if it exists
if 'doses_per_24_hrs' in df_clean.columns:
    df_clean = df_clean.withColumn(
        'doses_per_24_hrs_numeric',
        when(col('doses_per_24_hrs').isNotNull(), col('doses_per_24_hrs').cast(DoubleType())).otherwise(0.0)
    )
    numerical_features = ['total_drugs', 'doses_per_24_hrs_numeric']
else:
    numerical_features = ['total_drugs']

# Fill null values in numerical features
for feature in numerical_features:
    df_clean = df_clean.fillna({feature: 0.0})

print(f"   ✓ Numerical features prepared: {numerical_features}")

# String indexing for drug columns (create indices for each drug)
indexed_features = []
indexers = []

for drug_col in drug_columns[:3]:  # Use first 3 drug columns to keep feature space manageable
    if drug_col in df_clean.columns:
        # Fill null values with 'NONE'
        df_clean = df_clean.fillna({drug_col: 'NONE'})
        
        indexer = StringIndexer(
            inputCol=drug_col,
            outputCol=f"{drug_col}_index",
            handleInvalid="keep"
        )
        indexers.append(indexer)
        indexed_features.append(f"{drug_col}_index")

print(f"   ✓ Drug indexing configured for {len(indexers)} columns")

# Combine all features
feature_columns = numerical_features + indexed_features

print(f"   ✓ Total feature columns: {len(feature_columns)}")
print(f"\n   Feature list: {feature_columns}")

# Show class distribution
print("\n📊 Class Distribution:")
df_clean.groupBy("label").count().show()

print("\n" + "="*60)

🔄 Preprocessing data for MLlib models...
   Found 10 drug columns: ['drug1', 'drug2', 'drug3', 'drug4', 'drug5', 'drug6', 'drug7', 'drug8', 'drug9', 'drug10']
   ✓ Label encoding complete (safe=0, unsafe=1)
   ✓ Numerical features prepared: ['total_drugs', 'doses_per_24_hrs_numeric']
   ✓ Drug indexing configured for 3 columns
   ✓ Total feature columns: 5

   Feature list: ['total_drugs', 'doses_per_24_hrs_numeric', 'drug1_index', 'drug2_index', 'drug3_index']

📊 Class Distribution:
+-----+--------+
|label|   count|
+-----+--------+
|  0.0|20290631|
|  1.0|  191541|
+-----+--------+


+-----+--------+
|label|   count|
+-----+--------+
|  0.0|20290631|
|  1.0|  191541|
+-----+--------+


